In [ ]:
import time
start_time = time.time()

import pandas as pd
import torch
from datasets import load_dataset
from transformers import pipeline
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix

In [ ]:
if torch.backends.mps.is_available():
    device = "mps"
elif torch.cuda.is_available():
    device = 0
else:
    device = -1

device_name = "mps" if device == "mps" else ("cuda:0" if device == 0 else "cpu")
print(f"Selected device: {device_name}")

In [ ]:
model_name = "textattack/distilbert-base-uncased-MRPC"
classifier = pipeline(
    task="text-classification",
    model=model_name,
    tokenizer=model_name,
    device=device
)

label2id = classifier.model.config.label2id if hasattr(classifier.model.config, "label2id") else {"LABEL_0": 0, "LABEL_1": 1}
id2label = classifier.model.config.id2label if hasattr(classifier.model.config, "id2label") else {0: "not_equivalent", 1: "equivalent"}
print(f"Loaded model: {model_name}")
print(f"Label mapping: {id2label}")

In [ ]:
dataset = load_dataset("glue", "mrpc", split="validation")
print(f"Validation examples: {len(dataset)}")

preview_df = dataset.select(range(min(5, len(dataset)))).to_pandas()
print(preview_df[["sentence1", "sentence2", "label"]].to_string(index=False))

In [ ]:
batch_size = 32
predictions = []
prediction_scores = []
true_labels = []

for start_idx in range(0, len(dataset), batch_size):
    batch = dataset[start_idx:start_idx + batch_size]
    batch_inputs = [
        {"text": s1, "text_pair": s2}
        for s1, s2 in zip(batch["sentence1"], batch["sentence2"])
    ]
    outputs = classifier(batch_inputs, batch_size=batch_size, truncation=True)

    for out in outputs:
        label_name = out["label"]
        pred_id = label2id[label_name] if label_name in label2id else int(label_name.split("_")[-1])
        predictions.append(pred_id)
        prediction_scores.append(float(out["score"]))

    true_labels.extend(batch["label"])

print(f"Completed inference for {len(predictions)} examples.")

In [ ]:
accuracy = accuracy_score(true_labels, predictions)
precision, recall, f1, _ = precision_recall_fscore_support(
    true_labels,
    predictions,
    average="binary",
    zero_division=0
)
cm = confusion_matrix(true_labels, predictions)

print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1 Score:  {f1:.4f}")
print("Confusion Matrix:")
print(cm)

In [ ]:
results_df = pd.DataFrame([
    {
        "model_name": model_name,
        "dataset": "glue/mrpc",
        "split": "validation",
        "num_examples": len(dataset),
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "device": device_name,
        "batch_size": batch_size
    }
])

print(results_df.to_string(index=False))

In [ ]:
examples_df = dataset.to_pandas()[["sentence1", "sentence2", "label"]].copy()
examples_df = examples_df.rename(columns={"label": "true_label"})
examples_df["predicted_label"] = predictions
examples_df["prediction_confidence"] = prediction_scores
examples_df["true_label_name"] = examples_df["true_label"].map(id2label)
examples_df["predicted_label_name"] = examples_df["predicted_label"].map(id2label)
examples_df["correct"] = examples_df["true_label"] == examples_df["predicted_label"]

print(examples_df.head(10).to_string(index=False))

In [ ]:
mismatches_df = examples_df[~examples_df["correct"]].copy()
mismatch_count = len(mismatches_df)
match_count = len(examples_df) - mismatch_count

print(f"Matches: {match_count}")
print(f"Mismatches: {mismatch_count}")

error_summary_df = pd.DataFrame([
    {
        "true_0_pred_0": int(((examples_df["true_label"] == 0) & (examples_df["predicted_label"] == 0)).sum()),
        "true_0_pred_1": int(((examples_df["true_label"] == 0) & (examples_df["predicted_label"] == 1)).sum()),
        "true_1_pred_0": int(((examples_df["true_label"] == 1) & (examples_df["predicted_label"] == 0)).sum()),
        "true_1_pred_1": int(((examples_df["true_label"] == 1) & (examples_df["predicted_label"] == 1)).sum())
    }
])
print(error_summary_df.to_string(index=False))

if mismatch_count > 0:
    print(mismatches_df[["sentence1", "sentence2", "true_label", "predicted_label", "prediction_confidence"]].head(10).to_string(index=False))

In [ ]:
label_breakdown_df = (
    examples_df.groupby(["true_label", "predicted_label"]).size().reset_index(name="count")
    .sort_values(["true_label", "predicted_label"])
)
print(label_breakdown_df.to_string(index=False))

true_label_counts_df = examples_df["true_label"].value_counts().sort_index().rename_axis("true_label").reset_index(name="count")
pred_label_counts_df = examples_df["predicted_label"].value_counts().sort_index().rename_axis("predicted_label").reset_index(name="count")

print(true_label_counts_df.to_string(index=False))
print(pred_label_counts_df.to_string(index=False))

In [ ]:
elapsed_seconds = time.time() - start_time
print(f"Total runtime (seconds): {elapsed_seconds:.2f}")